In [ ]:
!pip install openai --upgrade

In [8]:
import requests
import json
import time
import os
import pandas as pd
import numpy as np
from datetime import datetime
from sentence_transformers import SentenceTransformer


In [ ]:
LISTING_URL = f"https://www.alphavantage.co/query?function=LISTING_STATUS&apikey={ALPHA_VANTAGE_KEY}"

DESC_FILE = "company_descriptions.json"
FINANCIALS_FILE = "company_financials.json"
PRICE_VOLUME_FILE = "company_price_volume.json"

# Load or download ticker list
if not os.path.exists("listing_status.csv"):
    print("Downloading full ticker list from Alpha Vantage...")
    response = requests.get(LISTING_URL)
    with open("listing_status.csv", "wb") as f:
        f.write(response.content)
    print("Ticker list downloaded and saved.")

tickers_df = pd.read_csv("listing_status.csv")
tickers_list = tickers_df['symbol'].unique().tolist()
print(f"Total tickers to process: {len(tickers_list)}")

# Helper: Load existing data
def load_existing_data(file_path):
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            data = json.load(f)
        symbols = {entry['symbol'] for entry in data}
        print(f"Loaded {len(data)} records from {file_path}.")
    else:
        data, symbols = [], set()
        print(f"No existing data in {file_path}.")
    return data, symbols

desc_data, desc_symbols = load_existing_data(DESC_FILE)
fin_data, fin_symbols = load_existing_data(FINANCIALS_FILE)
pv_data, pv_symbols = load_existing_data(PRICE_VOLUME_FILE)

# Robust Alpha Vantage API fetching
def fetch_json(url):
    response = requests.get(url)
    if response.status_code != 200:
        print(f"API Error ({response.status_code}) on URL: {url}")
        return None
    try:
        data = response.json()
        if "Note" in data or "Error Message" in data:
            print(f"API Limit/Error message received: {data}")
            return None
        return data
    except json.JSONDecodeError:
        print("Invalid JSON response.")
        return None

# Fetch description and financial metrics
def fetch_overview(symbol):
    url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={ALPHA_VANTAGE_KEY}"
    return fetch_json(url)

# Fetch historical price-volume data
def fetch_time_series(symbol):
    url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&outputsize=compact&apikey={ALPHA_VANTAGE_KEY}"
    return fetch_json(url)

processed = 0
total = len(tickers_list)

for ticker in tickers_list:
    processed += 1
    print(f"\nProcessing {ticker} ({processed}/{total})")

    # Fetch Overview (Descriptions & Financials)
    if ticker not in desc_symbols or ticker not in fin_symbols:
        overview = fetch_overview(ticker)
        if overview:
            # Save description
            if ticker not in desc_symbols and overview.get("Description"):
                desc_data.append({
                    "symbol": ticker,
                    "description": overview["Description"]
                })
                desc_symbols.add(ticker)
                print(f"✅ Description stored for {ticker}")

            # Save financial metrics
            financial_entry = {
                "symbol": ticker,
                "MarketCap": overview.get("MarketCapitalization"),
                "PERatio": overview.get("PERatio"),
                "DividendYield": overview.get("DividendYield"),
                "EBITDA": overview.get("EBITDA"),
                "RevenueTTM": overview.get("RevenueTTM"),
                "ProfitMargin": overview.get("ProfitMargin")
            }
            if ticker not in fin_symbols:
                fin_data.append(financial_entry)
                fin_symbols.add(ticker)
                print(f"✅ Financial metrics stored for {ticker}")
        else:
            print(f"⚠️ No overview data available for {ticker}")

        time.sleep(1)  # Safe delay to respect API rate limits

    # Fetch Price & Volume Patterns
    if ticker not in pv_symbols:
        ts_data = fetch_time_series(ticker)
        if ts_data and 'Time Series (Daily)' in ts_data:
            historical = ts_data['Time Series (Daily)']
            sorted_dates = sorted(historical.keys(), reverse=True)[:30]  # last 30 days
            price_volume = [{
                "date": date,
                "close": float(historical[date]['4. close']),
                "volume": int(historical[date]['5. volume'])
            } for date in sorted_dates]

            pv_data.append({
                "symbol": ticker,
                "price_volume_last_30_days": price_volume
            })
            pv_symbols.add(ticker)
            print(f"✅ Price-volume data stored for {ticker}")
        else:
            print(f"⚠️ No price-volume data for {ticker}")

        time.sleep(1)  # Safe delay

    # Checkpoint: Save after each ticker
    with open(DESC_FILE, "w") as f:
        json.dump(desc_data, f, indent=2)

    with open(FINANCIALS_FILE, "w") as f:
        json.dump(fin_data, f, indent=2)

    with open(PRICE_VOLUME_FILE, "w") as f:
        json.dump(pv_data, f, indent=2)

print("\n🎉 All available data fetched and stored successfully!")

Total tickers to process: 11942
Loaded 46 records from company_descriptions.json.
Loaded 37 records from company_financials.json.
Loaded 59 records from company_price_volume.json.

Processing A (1/11942)

Processing AA (2/11942)

Processing AAA (3/11942)
⚠️ No overview data available for AAA

Processing AAAU (4/11942)
⚠️ No overview data available for AAAU

Processing AACBR (5/11942)

Processing AACBU (6/11942)

Processing AACG (7/11942)

Processing AACI (8/11942)
⚠️ No overview data available for AACI

Processing AACIU (9/11942)

Processing AACIW (10/11942)
⚠️ No overview data available for AACIW

Processing AACT (11/11942)

Processing AACT-U (12/11942)

Processing AACT-WS (13/11942)

Processing AADR (14/11942)
⚠️ No overview data available for AADR

Processing AAL (15/11942)

Processing AALG (16/11942)
⚠️ No overview data available for AALG

Processing AAM (17/11942)

Processing AAM-U (18/11942)

Processing AAM-WS (19/11942)

Processing AAME (20/11942)

Processing AAMI (21/11942)

Pr

In [ ]:

desc_model = SentenceTransformer('all-MiniLM-L6-v2')

with open('company_descriptions.json') as f:
    descriptions = json.load(f)

for entry in descriptions:
    entry['embedding'] = desc_model.encode(entry['description']).tolist()

with open('company_descriptions_embedded.json', 'w') as f:
    json.dump(descriptions, f)


In [5]:
import requests
import json
import time
import numpy as np
import os
from sentence_transformers import SentenceTransformer


ALPHA_VANTAGE_KEY = "60G1A3ENUDKBDCO1"
TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", 
    "NVDA", "META", "NFLX", "INTC", "AMD"
]

DESCRIPTIONS_FILE = "company_descriptions.json"
EMBEDDINGS_FILE = "company_embeddings.json"

# Load existing descriptions if available
if os.path.exists(DESCRIPTIONS_FILE):
    with open(DESCRIPTIONS_FILE, "r") as f:
        company_data = json.load(f)
    symbols_fetched = {entry['symbol'] for entry in company_data}
    print(f"Loaded {len(company_data)} existing company descriptions.")
else:
    company_data = []
    symbols_fetched = set()
    print("No existing descriptions found; starting fresh.")

def get_description(symbol):
    url = (
        f"https://www.alphavantage.co/query?"
        f"function=OVERVIEW&symbol={symbol}&apikey={ALPHA_VANTAGE_KEY}"
    )
    print(f"Fetching description for {symbol}...")
    r = requests.get(url)
    if r.status_code != 200:
        print(f"Failed to fetch {symbol}: HTTP {r.status_code}")
        return None
    data = r.json()
    desc = data.get("Description", "")
    if not desc:
        print(f"No description found for {symbol}!")
    else:
        print(f"Description for {symbol}: {desc[:75]}...")
    return desc

# Fetch descriptions incrementally
for ticker in TICKERS:
    if ticker in symbols_fetched:
        print(f"Description for {ticker} already exists, skipping.")
        continue
    desc = get_description(ticker)
    if desc:
        company_data.append({"symbol": ticker, "description": desc})
        # Save after each description fetched
        with open(DESCRIPTIONS_FILE, "w") as f:
            json.dump(company_data, f, indent=2)
        print(f"Saved description for {ticker}.\n")
    time.sleep(15)  # Alpha Vantage free API limit (5 calls per minute)

# Initialize sentence-transformers model
print("\nLoading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded.\n")

# Generate embeddings
embeddings_data = []
for company in company_data:
    desc = company["description"]
    print(f"Embedding description for {company['symbol']}...")
    embedding = model.encode(desc).tolist()
    embeddings_data.append({
        "symbol": company["symbol"],
        "description": desc,
        "embedding": embedding
    })

# Save embeddings
with open(EMBEDDINGS_FILE, "w") as f:
    json.dump(embeddings_data, f, indent=2)
print(f"\nSaved embeddings for {len(embeddings_data)} companies to {EMBEDDINGS_FILE}.\n")

# Similarity function (cosine similarity)
def cosine_similarity(v1, v2):
    v1 = np.array(v1)
    v2 = np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Function to find similar companies
def find_similar(symbol, top_n=3):
    print(f"\nFinding top {top_n} companies similar to {symbol}...\n")
    with open(EMBEDDINGS_FILE, "r") as f:
        data = json.load(f)

    target = next((item for item in data if item["symbol"] == symbol), None)
    if not target:
        print(f"No embedding found for {symbol}.")
        return

    similarities = []
    for company in data:
        if company["symbol"] == symbol:
            continue
        similarity = cosine_similarity(target["embedding"], company["embedding"])
        similarities.append((company["symbol"], similarity))

    similarities.sort(key=lambda x: x[1], reverse=True)
    print(f"Top {top_n} similar companies to {symbol}:")
    for sym, sim_score in similarities[:top_n]:
        print(f"{sym}: Similarity {sim_score:.4f}")

# Example Usage:
find_similar("AAPL", top_n=3)


Loaded 10 existing company descriptions.
Description for AAPL already exists, skipping.
Description for MSFT already exists, skipping.
Description for GOOGL already exists, skipping.
Description for AMZN already exists, skipping.
Description for TSLA already exists, skipping.
Description for NVDA already exists, skipping.
Description for META already exists, skipping.
Description for NFLX already exists, skipping.
Description for INTC already exists, skipping.
Description for AMD already exists, skipping.

Loading embedding model...
Embedding model loaded.

Embedding description for AAPL...
Embedding description for MSFT...
Embedding description for GOOGL...
Embedding description for AMZN...
Embedding description for TSLA...
Embedding description for NVDA...
Embedding description for META...
Embedding description for NFLX...
Embedding description for INTC...
Embedding description for AMD...

Saved embeddings for 10 companies to company_embeddings.json.


Finding top 3 companies similar